In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs\\ablations_hpo"
FILE_NAME = f"M0_MOE_1s3w_hpo"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: M0_MOE_1s3w_hpo
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\ablations_hpo\M0_MOE_1s3w_hpo.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'moe_hpo_1s3w_hpo_v1'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 134 trials.
Best value (Accuracy): 0.9155
Best Hyperparameters:
  num_experts: 12
  utilization_ratio: 0.1
  MOE_routing_signal: linear_proj_input
  MOE_gate_temperature: 0.5
  MOE_use_shared_expert: True
  MOE_aux_coeff: 0.002034241475018406
  MOE_importance_coeff: 0.004180868723067521
  MOE_ctx_hidden_dim: 64
  MOE_ctx_out_dim: 16


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL

params_to_plot_MOE_CORE = [
    "num_experts", "utilization_ratio", "MOE_routing_signal", "MOE_use_shared_expert"
]

params_to_plot_MOE_AUX = [
    "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_importance_coeff", "MOE_gate_temperature", "MOE_aux_coeff"
]

In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [12]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
94,94,0.915547,None,2026-04-25 15:55:25.605818,0 days 05:09:39.046167
44,44,0.912214,None,2026-04-25 06:07:08.867809,0 days 07:24:43.575937
82,82,0.911693,None,2026-04-25 14:34:03.175981,0 days 05:07:15.833308
76,76,0.911693,None,2026-04-25 13:53:49.023708,0 days 05:35:33.768213
78,78,0.911615,None,2026-04-25 14:03:55.949638,0 days 05:18:54.843215
107,107,0.910911,None,2026-04-25 16:38:10.598057,0 days 05:31:38.101791
90,90,0.910182,None,2026-04-25 15:18:44.560837,0 days 03:45:06.103599
58,58,0.908281,None,2026-04-25 08:09:44.059800,0 days 07:04:57.539178
77,77,0.905365,None,2026-04-25 13:59:41.967350,0 days 05:29:58.660014
47,47,0.902917,None,2026-04-25 07:29:56.073730,0 days 07:09:16.602553


In [13]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'num_experts': 12, 'utilization_ratio': 0.1, 'MOE_routing_signal': 'linear_proj_input', 'MOE_gate_temperature': 0.5, 'MOE_use_shared_expert': True, 'MOE_aux_coeff': 0.002034241475018406, 'MOE_importance_coeff': 0.004180868723067521, 'MOE_ctx_hidden_dim': 64, 'MOE_ctx_out_dim': 16}
{'num_experts': 12, 'utilization_ratio': 0.1, 'MOE_routing_signal': 'linear_proj_input', 'MOE_gate_temperature': 0.5, 'MOE_use_shared_expert': True, 'MOE_aux_coeff': 0.001866930201759959, 'MOE_importance_coeff': 0.0032425517614221764, 'MOE_ctx_hidden_dim': 64, 'MOE_ctx_out_dim': 16}
{'num_experts': 12, 'utilization_ratio': 0.1, 'MOE_routing_signal': 'linear_proj_input', 'MOE_gate_temperature': 0.5, 'MOE_use_shared_expert': True, 'MOE_aux_coeff': 0.0011345362226656533, 'MOE_importance_coeff': 0.0010744583847151152, 'MOE_ctx_hidden_dim': 64, 'MOE_ctx_out_dim': 16}
{'num_experts': 12, 'utilization_ratio': 0.1, 'MOE_routing_signal': 'linear_proj_input', 'MOE_gate_temperature': 0.5, 'MOE_use_shared_expert': True,